# CBUS mini-reproduction: Kaggle CPU runner
Settings: **Internet ON**, accelerator **None**. Add your OpenRouter key as a Kaggle Secret named `OPENROUTER_API_KEY` (Add-ons > Secrets).
The authors' data and the LLM cache live in `/tmp` so they are **not** saved as notebook output (licensed texts).

## 1. Setup

In [ ]:
import os, subprocess
REPO = 'https://github.com/YOUR-USERNAME/cbus-repro'   # <- your repo
if not os.path.exists('/kaggle/working/cbus-repro'):
    subprocess.run(['git','clone',REPO,'/kaggle/working/cbus-repro'],check=True)
if not os.path.exists('/tmp/authors_data'):   # authors' release: link, do not re-publish
    subprocess.run(['git','clone','--depth','1','https://github.com/iai-group/emnlp2026-simreading','/tmp/authors_data'],check=True)
os.chdir('/kaggle/working/cbus-repro')
os.environ['SIMREADING_DATA'] = '/tmp/authors_data'
subprocess.run('pip install -q -r requirements.txt',shell=True)
print(os.getcwd())

In [ ]:
from kaggle_secrets import UserSecretsClient
os.environ['OPENROUTER_API_KEY'] = UserSecretsClient().get_secret('OPENROUTER_API_KEY')
print('key loaded:', bool(os.environ['OPENROUTER_API_KEY']))

## 2. Offline checks (free)

In [ ]:
!python scripts/01_check_metrics.py

In [ ]:
!python scripts/02_run_simulation.py --mode spr --mock --students 20 --cache /tmp/cache.jsonl --out-root /tmp/smoke_runs

## 3. Pilot (20 students) - inspect raw outputs and `meta.json` (drop rate, tokens) before scaling

In [ ]:
MODEL = 'meta-llama/llama-3.3-70b-instruct'
N = 20
for mode in ['baseline','spr']:
    !python scripts/02_run_simulation.py --mode {mode} --model {MODEL} --students {N} --workers 8 --cache /tmp/cache.jsonl

## 4. Main runs (set N = 300 after the pilot looks sane). Re-running resumes from the cache.

In [ ]:
N = 300
!python scripts/02_run_simulation.py --mode random --students {N}
for mode in ['baseline','spr','ts','spr_dropout']:
    !python scripts/02_run_simulation.py --mode {mode} --model {MODEL} --students {N} --workers 8 --cache /tmp/cache.jsonl

## 5. Evaluate

In [ ]:
!python scripts/03_evaluate.py runs/*-n300-s0 --plot results/score_hist.png

## 6. Clean up (do not leave licensed paraphrases in the saved output)

In [ ]:
import shutil
for p in ['/tmp/cache.jsonl','/tmp/authors_data','/tmp/smoke_runs']:
    shutil.rmtree(p, ignore_errors=True) if os.path.isdir(p) else (os.path.exists(p) and os.remove(p))
print('cleaned; only runs/ and results/ (aggregate CSV/PNG) remain')